# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. It demonstrates FAIR data standards and the use of Croissant schema for easy, reproducible analytics.

### Dataset Source
The dataset is described by a Croissant schema located at the URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

The dataset contains survey data on mental health indicators from residents of Kilifi County, Kenya, including demographic details, GAD-7, PHQ-9 scores, and more. We will reference entities by their Croissant `@id` throughout.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)

# Access metadata for overview
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
List available record sets, fields, and their Croissant `@id`s.

> All references use the `@id` from Croissant schema for full reproducibility.

Below, we retrieve available record sets and display their fields and columns, referencing by `@id`.

In [ ]:
# Retrieve available record sets
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f" - {rs['@id']}: {rs.get('name', 'No name')}")
    # List fields in the record set
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("   Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"      {field['@id']} (name: {field.get('name', '')})")
        else:
            print(f"      {field}")
    # List columns (for tabular data)
    columns = rs.get('column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        print("   Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"      {col['@id']} (name: {col.get('name', '')})")
            else:
                print(f"      {col}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the record set and field `@id` values from the overview above.

In [ ]:
# Build list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
print("Extracting data for each record set:")

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  - {record_set_id}: {df.shape[0]} records, {df.shape[1]} columns")
    except Exception as e:
        print(f"  - {record_set_id}: Error loading records ({e})")

# Pick the main record set for exploration:
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]  # Select first (or set manually)
    print(f"\nFields in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, group by attributes.

Refer to fields by their Croissant `@id`. 

For demonstration, we'll use PHQ-9 score (`phq9_score`), GAD-7 score (`gad7_score`) or similar numeric fields if present, and demographic as group field.

In [ ]:
# Select fields by @id
df = dataframes[main_record_set_id]
# Detect numeric fields (example: scores)
numeric_field_candidates = [col for col in df.columns if 'score' in col or 'phq9' in col or 'gad7' in col or 'psq' in col]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.select_dtypes(include='number').columns[0]

# Set threshold for filtering
threshold = 10
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by demographic field, e.g. 'age' or 'level_of_education'
group_field_candidates = [col for col in df.columns if 'age' in col or 'education' in col or 'gender' in col or 'marital' in col]
group_field = group_field_candidates[0] if group_field_candidates else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the dataset using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(7,4))
df[numeric_field].dropna().hist(bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("count")
plt.show()

# Boxplot by group
if group_field:
    plt.figure(figsize=(7,4))
    df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We accessed record sets, filtered and normalized data, grouped by demographic attributes, and visualized results. The use of Croissant `@id` ensures reproducibility and clear referencing of dataset entities.

Key findings may include distributions of PHQ-9 and GAD-7 scores, differences by demographic groups, and insights for community health strategy.

Next steps could include more detailed statistical analysis, model training, or AI-driven insights using the standardized dataset.